# Favorite World — Unlimited Free Cloud Voiceover Server (Google Colab T4)

Runs **Resemble AI Chatterbox Multilingual (Hindi + English + Voice Cloning)** 100% on **Google's Free Cloud GPU** — zero load on your PC.

### Steps:
1. In Colab menu: **Runtime → Change runtime type → T4 GPU → Save**.
2. Click **Run All**.
3. Copy the printed `https://xxxx.gradio.live` link and paste it into **Voiceover Studio Pro → Optional Cloud Boost → Custom Colab / Gradio URL** (or pass `--space-url` to `voice_gen.py`).

In [ ]:
!pip install -q --no-deps chatterbox-tts
!pip install -q resemble-perth conformer==0.3.2 s3tokenizer omegaconf pykakasi==2.3.0 pyloudnorm soundfile librosa==0.11.0


In [ ]:
import sys, types, tempfile
for mod in ("pkuseg", "spacy_pkuseg"):
    try:
        __import__(mod)
    except Exception:
        sys.modules[mod] = types.ModuleType(mod)

import torch
import soundfile as sf
import numpy as np
import gradio as gr
from chatterbox.mtl_tts import ChatterboxMultilingualTTS

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using cloud device:", device)
model = ChatterboxMultilingualTTS.from_pretrained(device=device)

def generate_tts_audio(text_input, language_id, audio_prompt_path_input, exaggeration_input, temperature_input, seed_num_input, cfgw_input):
    if int(seed_num_input or 0) > 0:
        torch.manual_seed(int(seed_num_input))
    kwargs = dict(
        language_id=language_id or "hi",
        exaggeration=float(exaggeration_input),
        temperature=float(temperature_input),
        cfg_weight=float(cfgw_input),
    )
    if audio_prompt_path_input:
        kwargs["audio_prompt_path"] = audio_prompt_path_input
    wav = model.generate(text_input, **kwargs)
    arr = wav.squeeze().detach().cpu().numpy().astype(np.float32)
    out = tempfile.NamedTemporaryFile(suffix=".wav", delete=False).name
    sf.write(out, arr, int(model.sr))
    return out

with gr.Blocks() as demo:
    gr.Markdown("# Favorite World Cloud Voiceover (Chatterbox Multilingual)")
    txt = gr.Textbox(label="Text")
    lang = gr.Dropdown(["hi", "en"], value="hi", label="Language")
    ref = gr.Audio(type="filepath", label="Reference Audio (Optional)")
    ex = gr.Slider(0.25, 1.2, value=0.55, label="Exaggeration")
    temp = gr.Slider(0.2, 1.2, value=0.75, label="Temperature")
    seed = gr.Number(value=0, label="Seed")
    cfg = gr.Slider(0.2, 1.0, value=0.5, label="CFG/Pace")
    btn = gr.Button("Generate")
    out = gr.Audio(type="filepath", label="Output")
    btn.click(generate_tts_audio, [txt, lang, ref, ex, temp, seed, cfg], [out], api_name="generate_tts_audio")

demo.queue().launch(share=True, debug=True)
